# Learning and compute

This notebook answers three separate questions:

1. **Does the model learn?** Plot validation NLL and the task's primary success measure over optimizer steps.
2. **Do additional passes refine the prediction?** Inspect per-pass loss without treating correlated passes as independent runs.
3. **What does training cost?** Compare measured throughput only among runs using the same task, device, batch size, and evaluation contract.

Faint lines are individual seeds; heavy lines are unsmoothed medians at observed checkpoints. The notebook never saves figures automatically—uncomment a `savefig` line only for a figure you want to keep.

In [ ]:
from collections import defaultdict
from pathlib import Path
from statistics import median
import sys

import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "experiments").exists() else HERE.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from figures.plotting_utils import (
    ARCHITECTURE_COLORS,
    filter_records,
    grouped,
    load_training_records,
    metric_label,
    plot_seed_and_median_curves,
    primary_metric,
    set_plot_style,
    unique_values,
)

set_plot_style()
RESULT_ROOT = REPO_ROOT / "results"
FIGURE_DIR = REPO_ROOT / "figures"


In [ ]:
records = load_training_records(RESULT_ROOT)
print(f"Loaded {len(records)} evaluation checkpoints from {RESULT_ROOT}")
print("tasks:", unique_values(records, "task"))
print("architectures:", unique_values(records, "architecture"))
print("devices:", unique_values(records, "device"))
print("seeds:", unique_values(records, "seed"))

if not records:
    print("No metrics.jsonl artifacts found. Point RESULT_ROOT at a completed result tree.")

## Select a comparable slice

Keep `DEVICE` explicit when comparing throughput. For learning curves it may be left as `None`. If multiple pilots reused the same task/architecture/seed, narrow `RESULT_ROOT` to the intended timestamped experiment so they are not silently pooled.

In [ ]:
TASK = "shortest_path"  # pointer_chasing, tracking, permutation, state_machine, random_graph_walk, othello
DEVICE = None           # set to "mps", "cuda", or "cpu" for compute comparisons
ARCHITECTURES = [
    "transformer",
    "memory_tape",
    "joint_memory_tape",
    "memory_concat",
    "memory_update",
]
SUCCESS_METRIC = primary_metric(TASK)

selected = filter_records(records, task=TASK, device=DEVICE)
selected = [row for row in selected if row.get("architecture") in ARCHITECTURES]
print(f"Selected {len(selected)} checkpoints; primary metric = {SUCCESS_METRIC!r}")
print("runs:", len({row["run_dir"] for row in selected}))
run_keys = defaultdict(set)
for row in selected:
    run_keys[(row.get("architecture"), row.get("seed"))].add(row["run_dir"])
duplicates = {key: paths for key, paths in run_keys.items() if len(paths) > 1}
if duplicates:
    print("WARNING: multiple runs share an architecture/seed key; narrow RESULT_ROOT before interpreting medians:")
    for key, paths in duplicates.items():
        print(" ", key, *sorted(paths), sep="\n    ")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
plot_seed_and_median_curves(axes[0], selected, metric="loss")
axes[0].set_title("Validation likelihood")

plot_seed_and_median_curves(axes[1], selected, metric=SUCCESS_METRIC)
axes[1].set_title(f"Task success: {metric_label(SUCCESS_METRIC)}")
axes[1].set_ylim(-0.02, 1.02)

if any(row.get("level") is not None for row in selected):
    plot_seed_and_median_curves(axes[2], selected, metric="level")
    axes[2].set_title("BBH curriculum level")
else:
    secondary = "valid_edge_rate" if TASK == "shortest_path" else "sequence_legality"
    plot_seed_and_median_curves(axes[2], selected, metric=secondary)
    axes[2].set_title(metric_label(secondary))
    axes[2].set_ylim(-0.02, 1.02)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=min(5, len(labels)), bbox_to_anchor=(0.5, -0.04))
for ax in axes:
    legend = ax.get_legend()
    if legend:
        legend.remove()
fig.suptitle(f"{TASK}: learning curves", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_learning.png", dpi=220, bbox_inches="tight")


## Pass refinement and gradient routing

The next figure intentionally shows **one run at a time**. Per-pass losses share weights, data, and activations; error bands across passes would falsely suggest independent observations. A useful refinement signature is falling loss over trained passes without vanishing memory-writer/reader gradients.

In [ ]:
MULTIPASS_ARCHITECTURE = "memory_tape"
SEED = 1337
candidate_runs = sorted({
    row["run_dir"] for row in selected
    if row.get("architecture") == MULTIPASS_ARCHITECTURE and row.get("seed") == SEED
})
RUN_DIR = candidate_runs[-1] if candidate_runs else None
run_rows = sorted([row for row in selected if row["run_dir"] == RUN_DIR], key=lambda row: row["step"])
print("run:", RUN_DIR)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.1))
if not run_rows:
    for ax in axes:
        ax.text(0.5, 0.5, "No selected multi-pass run", ha="center", va="center")
        ax.set_axis_off()
else:
    pass_keys = sorted(
        {key for row in run_rows for key in row if key.startswith("pass_") and key.endswith("_loss")},
        key=lambda key: int(key.split("_")[1]),
    )
    for pass_key in pass_keys:
        points = [(row["step"], row[pass_key]) for row in run_rows if row.get(pass_key) is not None]
        axes[0].plot(*zip(*points), label=pass_key.replace("_", " "))
    axes[0].set(title="Loss by recurrent pass", xlabel="Optimizer step", ylabel="NLL")
    axes[0].legend()

    gradient_keys = [
        "gradient_backbone_mean",
        "gradient_memory_writer_mean",
        "gradient_memory_attention_mean",
        "gradient_memory_gate_mean",
    ]
    for key in gradient_keys:
        points = [(row["step"], row[key]) for row in run_rows if row.get(key) is not None]
        if points:
            axes[1].plot(*zip(*points), label=key.removeprefix("gradient_").removesuffix("_mean"))
    axes[1].set(title="Mean gradient norm by subsystem", xlabel="Optimizer step", ylabel="L2 norm")
    axes[1].set_yscale("log")
    axes[1].legend()

    gate_points = [
        (row["step"], row["memory_gate_mean_abs_effective"])
        for row in run_rows if row.get("memory_gate_mean_abs_effective") is not None
    ]
    if gate_points:
        axes[2].plot(*zip(*gate_points), color=ARCHITECTURE_COLORS["memory_tape"])
        axes[2].set(title="Effective memory-gate magnitude", xlabel="Optimizer step", ylabel="Mean |gate|")
    else:
        axes[2].text(0.5, 0.5, "Architecture has no scalar gate", ha="center", va="center")
        axes[2].set_axis_off()
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_{MULTIPASS_ARCHITECTURE}_mechanics.png", dpi=220, bbox_inches="tight")


## Measured training throughput

Throughput is a systems measurement, not an architecture-invariant property. Compare it only after fixing device, batch size, sequence length/task, precision, and software environment. Each dot below is a run's final logged window; medians summarize seeds without hiding them.

In [ ]:
final_by_run = []
for (_run_dir,), rows in grouped(selected, "run_dir").items():
    rows = sorted(rows, key=lambda row: row["step"])
    if rows and rows[-1].get("train_tok_per_s") is not None:
        final_by_run.append(rows[-1])

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for index, architecture in enumerate(ARCHITECTURES):
    values = [row["train_tok_per_s"] for row in final_by_run if row.get("architecture") == architecture]
    if not values:
        continue
    offsets = [(item - (len(values) - 1) / 2) * 0.07 for item in range(len(values))]
    ax.scatter([index + offset for offset in offsets], values, color=ARCHITECTURE_COLORS.get(architecture), alpha=0.65)
    ax.hlines(median(values), index - 0.28, index + 0.28, color="black", linewidth=2)
ax.set_xticks(range(len(ARCHITECTURES)), [name.replace("_", "\n") for name in ARCHITECTURES])
ax.set_ylabel("Training tokens / second")
ax.set_title(f"{TASK}: final measured throughput ({DEVICE or 'mixed devices—set DEVICE'})")
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_train_throughput.png", dpi=220, bbox_inches="tight")


## Parameter and observed memory footprint

Parameter count is exact. Process peak RSS is a process-wide high-water mark and may include Python, data, and allocator state; accelerator memory is backend-reported and should not be equated with recurrent-tape bytes. These panels are therefore complementary rather than interchangeable efficiency measures.

In [ ]:
resource_fields = [
    ("non_embedding_parameters", 1e6, "Non-embedding parameters (millions)"),
    ("process_peak_rss_bytes", 1024 ** 3, "Process peak RSS (GiB)"),
]
accelerator_candidates = [
    ("cuda_max_memory_allocated_bytes", "CUDA max allocated (GiB)"),
    ("mps_driver_allocated_bytes", "MPS driver allocated (GiB)"),
    ("mps_current_allocated_bytes", "MPS current allocated (GiB)"),
]
accelerator = next(((key, label) for key, label in accelerator_candidates
                    if any(row.get(key) is not None for row in final_by_run)), None)
if accelerator:
    resource_fields.append((accelerator[0], 1024 ** 3, accelerator[1]))

fig, axes = plt.subplots(1, len(resource_fields), figsize=(5 * len(resource_fields), 4.3))
if len(resource_fields) == 1:
    axes = [axes]
for ax, (field, scale, label) in zip(axes, resource_fields):
    for index, architecture in enumerate(ARCHITECTURES):
        values = [row[field] / scale for row in final_by_run
                  if row.get("architecture") == architecture and row.get(field) is not None]
        if values:
            ax.scatter([index] * len(values), values, color=ARCHITECTURE_COLORS.get(architecture), alpha=0.6)
            ax.hlines(median(values), index - 0.25, index + 0.25, color="black", linewidth=2)
    ax.set_xticks(range(len(ARCHITECTURES)), [name.replace("_", "\n") for name in ARCHITECTURES])
    ax.set_ylabel(label)
fig.suptitle(f"{TASK}: size and observed resource footprint", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_resource_footprint.png", dpi=220, bbox_inches="tight")
